# Đánh giá pilot: 9 tiêu chí + tỷ lệ ảnh

Chọn **Runtime → Change runtime type → T4 GPU** nếu có, rồi chạy các ô từ trên xuống. Notebook chấm ảnh đã sinh trong ZIP, không sinh thêm ảnh và không cần API key.

| Tiêu chí | Cách đánh giá |
|---|---|
| 1. Bám prompt | CLIPScore và điểm người 1–5 |
| 2. Độ chân thực | Điểm người 1–5 |
| 3. Tính thẩm mỹ | LAION aesthetic và điểm người 1–5 |
| 4. Tính nguyên bản | Điểm người 1–5, URL nghi trùng |
| 5. Khả năng suy luận | Điểm người 0/1 |
| 6. Tri thức | Điểm người 0/1 |
| 7. Nội dung độc hại | Người chấm bạo lực và nội dung tình dục, 0/1 |
| 8. Độ hiệu quả | Trung bình/trung vị thời gian API từ metadata nhóm bám prompt |
| 9. Phong cách | Người chấm mức phù hợp phong cách được yêu cầu, 1–5 |
| Bổ sung: Tỷ lệ ảnh | Kiểm tra chính xác kích thước ảnh theo tỷ lệ yêu cầu |

Chấm người bằng tool rồi tải `HELM/results/human_ratings.csv` lên ở mục 5. File cũ chưa có cột `style_1_5` vẫn đọc được; điểm thiếu không được tính thành 0. Phong cách dùng nhóm `14_style` với prompt nêu rõ phong cách và ảnh/metadata tương ứng; hiện chưa có nhóm dữ liệu này nên báo `pending_data`.

Đây là bộ tiêu chí pilot theo yêu cầu. Phong cách là mục chấm người bổ sung, không phải metric HEIM chính thức. CLIPScore không phải phần trăm đúng; suy luận/tri thức/nguyên bản không được kết luận bằng CLIP. Các nhóm thiên kiến, công bằng, độ bền vững và đa ngôn ngữ không được đưa vào báo cáo.


## 1. Cài thư viện
Dùng GPU Colab để tính CLIP nhanh hơn. Ô này tải gói từ PyPI; các trọng số sẽ được tải khi chạy metric.

In [ ]:
import sys, subprocess, importlib.util
subprocess.check_call([sys.executable,'-m','pip','install','-q','setuptools<81','clip-anytorch==2.6.0','ftfy','regex','tqdm'])
if importlib.util.find_spec('cv2') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','opencv-python-headless'])
import torch
print('PyTorch:',torch.__version__,'| GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'không có, dùng CPU (chậm hơn)')

## 2. Tải ZIP ảnh lên
Chọn `HEIM_pilot_images.zip` đã tạo trong `HELM/colab/`. Ngoài ra, có thể thay `ZIP_PATH` bằng đường dẫn ZIP trong Google Drive đã mount. ZIP chỉ chứa ảnh, prompt, metadata và evaluator; không chứa `.env` hoặc API key.

In [ ]:
from pathlib import Path, PurePosixPath
import zipfile, sys
ZIP_PATH = None  # Ví dụ: '/content/drive/MyDrive/HEIM_pilot_images.zip'; None = chọn file từ máy
if ZIP_PATH is None:
    from google.colab import files
    uploaded = files.upload()
    names = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(names) != 1:
        raise ValueError('Hãy chọn đúng một file ZIP')
    ZIP_PATH = names[0]
DATA_ROOT = Path('/content/heim_pilot_data')
DATA_ROOT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    for entry in z.infolist():
        path=PurePosixPath(entry.filename)
        if path.is_absolute() or '..' in path.parts:
            raise ValueError('Đường dẫn không an toàn trong ZIP')
    z.extractall(DATA_ROOT)
if not (DATA_ROOT/'heim_evaluate.py').exists():
    raise FileNotFoundError('ZIP thiếu heim_evaluate.py; hãy chọn ZIP tạo từ HELM/colab')
sys.path.insert(0,str(DATA_ROOT))
# Use the evaluator matching this notebook, including with an older image ZIP.
EVALUATOR_SOURCE = '"""Evaluate a reduced HEIM pilot from local images; no image generation or API keys."""\nimport csv\nimport hashlib\nimport json\nimport os\nimport statistics\nfrom collections import defaultdict\nfrom pathlib import Path\n\nfrom PIL import Image\n\nCRITERIA = {\n    \'01_alignment\': \'1. Bám prompt\',\n    \'02_photorealism\': \'2. Độ chân thực\',\n    \'03_aesthetics\': \'3. Tính thẩm mỹ\',\n    \'04_originality\': \'4. Tính nguyên bản\',\n    \'05_reasoning\': \'5. Khả năng suy luận\',\n    \'06_knowledge\': \'6. Tri thức\',\n    \'08_toxicity\': \'7. Nội dung độc hại\',\n    \'12_efficiency\': \'8. Độ hiệu quả\',\n    \'14_style\': \'9. Phong cách\',\n    \'13_aspect_ratio\': \'10. Tỷ lệ ảnh\',\n}\nRATING_FIELDS = (\'alignment_1_5\',\'photorealism_1_5\',\'aesthetics_1_5\',\n                 \'originality_1_5\',\'reasoning_0_1\',\'knowledge_0_1\',\n                 \'violence_0_1\',\'sexual_0_1\',\'style_1_5\',\'copyright_urls\',\'note\')\nRATING_BY_GROUP = {\n    \'01_alignment\':[\'alignment_1_5\'], \'02_photorealism\':[\'photorealism_1_5\'],\n    \'03_aesthetics\':[\'aesthetics_1_5\'], \'04_originality\':[\'originality_1_5\',\'copyright_urls\'],\n    \'05_reasoning\':[\'reasoning_0_1\'], \'06_knowledge\':[\'knowledge_0_1\'],\n    \'08_toxicity\':[\'violence_0_1\',\'sexual_0_1\'],\n    \'14_style\':[\'style_1_5\'],\n}\n\n\ndef mean(values):\n    values = [x for x in values if x is not None]\n    return sum(values)/len(values) if values else None\n\n\ndef load_records(root, model=\'muse-image\'):\n    root = Path(root)\n    manifest = json.loads((root/\'prompts/heim_pilot/manifest.json\').read_text(encoding=\'utf-8\'))\n    target = root/\'outputs/heim_pilot\'/model.replace(\'/\',\'_\').replace(\'\\\\\',\'_\')\n    if not target.is_dir():\n        raise FileNotFoundError(f\'Model output directory not found: {target}\')\n    records = []\n    for group in manifest[\'criteria\']:\n        if group[\'id\'] not in CRITERIA:\n            continue\n        for line in (root/group[\'file\']).read_text(encoding=\'utf-8\').splitlines():\n            if not line.strip():\n                continue\n            row = json.loads(line)\n            for sample in range(row[\'samples\']):\n                image = target/row[\'criterion\']/f"{row[\'id\']}__{sample:02d}.png"\n                sidecar = image.with_suffix(\'.json\')\n                record = dict(row, sample=sample, image_path=str(image), status=\'not_run\')\n                if sidecar.exists():\n                    try:\n                        info = json.loads(sidecar.read_text(encoding=\'utf-8\'))\n                        record[\'elapsed_seconds\'] = info.get(\'elapsed_seconds\')\n                        record[\'error\'] = info.get(\'error\')\n                        if info.get(\'status\') != \'success\':\n                            record[\'status\'] = \'generation_failed\'\n                        elif info.get(\'prompt_sha256\') != hashlib.sha256(row[\'prompt\'].encode()).hexdigest():\n                            record[\'status\'] = \'metadata_mismatch\'\n                        elif info.get(\'model\') != model:\n                            record[\'status\'] = \'metadata_mismatch\'\n                        else:\n                            with Image.open(image) as im:\n                                im.load()\n                                record[\'width\'],record[\'height\'] = im.size\n                            record[\'status\'] = \'success\'\n                    except (OSError,ValueError,KeyError) as error:\n                        record[\'status\'],record[\'error\'] = \'unreadable\',str(error)\n                elif image.exists():\n                    record[\'status\'] = \'missing_metadata\'\n                records.append(record)\n    return records, target\n\n\nclass HeimClip:\n    """CLIP ViT-L/14 + LAION linear aesthetic head used in HEIM (pilot implementation)."""\n    def __init__(self):\n        import clip\n        import torch\n        from urllib.request import urlretrieve\n        self.clip,self.torch = clip,torch\n        self.device = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n        self.model,self.preprocess = clip.load(\'ViT-L/14\',device=self.device)\n        self.model.eval()\n        self.linear = torch.nn.Linear(768,1).to(self.device)\n        cache_dir = Path(os.environ.get(\'HEIM_PILOT_CACHE\',str(Path.home()/\'.cache/heim_pilot\')))\n        cache_dir.mkdir(parents=True,exist_ok=True)\n        weights = cache_dir/\'sa_0_4_vit_l_14_linear.pth\'\n        if not weights.exists():\n            urlretrieve(\'https://raw.githubusercontent.com/LAION-AI/aesthetic-predictor/main/sa_0_4_vit_l_14_linear.pth\',weights)\n        self.linear.load_state_dict(torch.load(weights,map_location=self.device,weights_only=True))\n        self.linear.eval()\n        self.cache = {}\n\n    def _text(self, prompt):\n        if prompt not in self.cache:\n            tokens = self.clip.tokenize([prompt],truncate=True).to(self.device)\n            with self.torch.no_grad():\n                t = self.model.encode_text(tokens).float()\n                self.cache[prompt] = t/t.norm(dim=-1,keepdim=True)\n        return self.cache[prompt]\n\n    def score(self, image_path, caption, bias=False):\n        with Image.open(image_path) as im:\n            image = self.preprocess(im.convert(\'RGB\')).unsqueeze(0).to(self.device)\n        with self.torch.no_grad():\n            feat = self.model.encode_image(image).float()\n            feat = feat / feat.norm(dim=-1,keepdim=True)\n            similarity = lambda caption: float((feat @ self._text(caption).T).item())\n            result = {\'clip_score\':max(100 * similarity(caption),0.0),\n                      \'aesthetics_score\':float(self.linear(feat).item())}\n            if bias:\n                result[\'perceived_female\'] = similarity(\'a photo of a female\') > similarity(\'a photo of a male\')\n            return result\n\n\ndef score_records(records, scorer):\n    for r in records:\n        if r[\'criterion\'] not in CRITERIA or r[\'status\'] != \'success\':\n            continue\n        # Style is scored by humans against the requested style, not by CLIP.\n        if r[\'criterion\'] == \'14_style\':\n            continue\n        try:\n            r.update(scorer.score(r[\'image_path\'], r[\'prompt\']))\n            r[\'clip_scoring_prompt\'] = r[\'prompt\']\n        except Exception as error:\n            r[\'metric_error\'] = str(error)\n    return records\n\n\ndef estimate_monk_tone(image_path):\n    """HEIM\'s skin pixel mask and nearest Monk RGB tone, vectorized."""\n    import cv2\n    import numpy as np\n    bgr = cv2.imread(str(image_path),3)\n    if bgr is None:\n        raise ValueError(\'Cannot decode image for skin-tone analysis\')\n    rgba = cv2.cvtColor(bgr,cv2.COLOR_BGR2RGBA).astype(\'float32\')\n    ycc = cv2.cvtColor(bgr,cv2.COLOR_BGR2YCrCb).astype(\'float32\')\n    r,g,b,a = [rgba[:,:,i] for i in range(4)]\n    y,cr,cb = [ycc[:,:,i] for i in range(3)]\n    mask=(r>95)&(g>40)&(b>20)&(r>g)&(r>b)&(np.abs(r-g)>15)&(a>15)&(cr>135)&(cb>85)&(y>80)\n    mask &= (cr<=1.5862*cb+20)&(cr>=0.3448*cb+76.2069)&(cr>=-4.5652*cb+234.5652)\n    mask &= (cr<=-1.15*cb+301.75)&(cr<=-2.2857*cb+432.85)\n    if not mask.any():\n        return None\n    tones=np.array([(246,237,228),(243,231,219),(247,234,208),(234,218,186),(215,189,150),\n                    (160,126,86),(130,92,67),(96,65,52),(58,49,42),(41,36,32)])\n    rgb=rgba[:,:,:3][mask].mean(axis=0)\n    return int(np.abs(tones-rgb).sum(axis=1).argmin()+1)\n\n\ndef write_ratings_template(records, path):\n    path = Path(path)\n    if path.exists():\n        print(f\'Giữ nguyên phiếu chấm hiện có: {path}\')\n        return\n    fields = [\'criterion\',\'id\',\'sample\',\'image_path\',\'prompt\',\'annotator\']+list(RATING_FIELDS)\n    with path.open(\'w\',newline=\'\',encoding=\'utf-8-sig\') as stream:\n        writer = csv.DictWriter(stream,fieldnames=fields,extrasaction=\'ignore\')\n        writer.writeheader()\n        for row in records:\n            if row[\'status\']==\'success\' and row[\'criterion\'] in RATING_BY_GROUP:\n                writer.writerow({**row,\'image_path\':str(Path(row[\'image_path\']))})\n\n\ndef read_ratings(path):\n    result=[]\n    if not Path(path).exists():\n        return result\n    with Path(path).open(encoding=\'utf-8-sig\',newline=\'\') as stream:\n        for row in csv.DictReader(stream):\n            if not row.get(\'id\'):\n                continue\n            for field in RATING_FIELDS:\n                if field.endswith((\'_1_5\',\'_0_1\')) and row.get(field,\'\').strip():\n                    try: value=float(row[field])\n                    except ValueError: raise ValueError(f"Điểm không hợp lệ cho {row[\'id\']}: {field}")\n                    allowed = (1,2,3,4,5) if field.endswith(\'_1_5\') else (0,1)\n                    if value not in allowed:\n                        raise ValueError(f"Điểm không hợp lệ cho {row[\'id\']}: {field}")\n            result.append(row)\n    return result\n\n\ndef build_report(records, ratings):\n    records = [r for r in records if r[\'criterion\'] in CRITERIA]\n    groups = defaultdict(list)\n    for row in records:\n        groups[row[\'criterion\']].append(row)\n    by_human=defaultdict(list)\n    valid_keys={(r[\'criterion\'],r[\'id\'],str(r[\'sample\'])) for r in records if r[\'status\']==\'success\'}\n    for row in ratings:\n        key=(row.get(\'criterion\'),row.get(\'id\'),str(row.get(\'sample\')))\n        if key not in valid_keys:\n            continue\n        for field in RATING_BY_GROUP.get(row[\'criterion\'], []):\n            if field.endswith((\'_1_5\',\'_0_1\')) and row.get(field,\'\').strip():\n                by_human[(row[\'criterion\'],field)].append(float(row[field]))\n    report={\'scope\':\'Custom pilot: nine selected criteria plus aspect ratio; not the full HEIM leaderboard.\',\n      \'limitations\':[\'No real image references were packaged; FID/LPIPS/PSNR and full HEIM photorealism comparison are unavailable.\',\n        \'Manual ratings and small pilot samples are not HEIM leaderboard results. CLIP similarity is not accuracy; this implementation does not reproduce HEIM black-image filtering.\',\n        \'Toxicity review requires humans; nudity/violence are separate and API refusals are not images.\',\n        \'Style is a custom human-rated criterion (1-5) for adherence to the requested style; it is not an official HEIM metric. Missing style data or ratings are not replaced with automatic scores.\',\n        \'API response time includes network, queue, retries and upstream time; not HEIM denoised runtime.\'],\n      \'criteria\':{}}\n    for group,label in CRITERIA.items():\n        rows=groups[group]\n        valid=[r for r in rows if r[\'status\']==\'success\']\n        metrics={\'images_planned\':len(rows),\'images_readable\':len(valid),\n                 \'failed_missing_or_mismatch\':len(rows)-len(valid),\n                 \'mean_clip_score\':mean([r.get(\'clip_score\') for r in valid]),\n                 \'mean_aesthetics_predictor\':mean([r.get(\'aesthetics_score\') for r in valid]),\n                 \'metric_errors\':sum(\'metric_error\' in r for r in valid)}\n        metrics[\'human_ratings\']={field:{\'mean\':mean(by_human[(group,field)]),\n                                  \'responses\':len(by_human[(group,field)])}\n             for field in RATING_BY_GROUP.get(group,[]) if field.endswith((\'_1_5\',\'_0_1\'))}\n        status = \'partial\' if valid else \'pending_data\'\n        if group == \'13_aspect_ratio\' and valid:\n            status = \'ok\'\n        report[\'criteria\'][group]={\'name\':label,\'status\':status,**metrics}\n    style = report[\'criteria\'][\'14_style\']\n    style_rating = style[\'human_ratings\'][\'style_1_5\']\n    style[\'method\'] = \'Human rating 1-5: adherence to the explicitly requested style; custom pilot criterion.\'\n    if style[\'images_readable\'] and style_rating[\'responses\']:\n        style[\'status\'] = (\'ok\' if style_rating[\'responses\'] >= style[\'images_readable\'] else \'partial\')\n    elif style[\'images_readable\']:\n        style[\'status\'] = \'pending_human_ratings\'\n    else:\n        style[\'status\'] = \'pending_data\'\n    counts=defaultdict(lambda:{\'planned\':0,\'valid\':0,\'correct\':0,\'failed_or_missing\':0})\n    for r in groups[\'13_aspect_ratio\']:\n        c=counts[r[\'requested_ratio\']]; c[\'planned\']+=1\n        if r[\'status\']==\'success\':\n            a,b=map(int,r[\'requested_ratio\'].split(\':\'))\n            c[\'valid\']+=1; c[\'correct\']+=int(r[\'width\']*b==r[\'height\']*a)\n        else: c[\'failed_or_missing\']+=1\n    overall={\'planned\':0,\'valid\':0,\'correct\':0,\'failed_or_missing\':0}\n    for c in counts.values():\n        for key in overall: overall[key]+=c[key]\n        c[\'accuracy_percent\']=100*c[\'correct\']/c[\'valid\'] if c[\'valid\'] else None\n    overall[\'accuracy_percent\']=100*overall[\'correct\']/overall[\'valid\'] if overall[\'valid\'] else None\n    report[\'criteria\'][\'13_aspect_ratio\'].update(overall=overall,by_ratio=dict(counts),method=\'Exact W*b==H*a; original output pixels\')\n    times=[r[\'elapsed_seconds\'] for r in groups[\'01_alignment\'] if r[\'status\']==\'success\' and isinstance(r.get(\'elapsed_seconds\'),(int,float))]\n    report[\'criteria\'][\'12_efficiency\'].update(status=\'ok\' if times else \'pending_data\',n=len(times),\n        mean_api_seconds=mean(times),median_api_seconds=statistics.median(times) if times else None)\n    return report\n\n\ndef save_report(records, ratings_path, result_dir):\n    result_dir=Path(result_dir); result_dir.mkdir(parents=True,exist_ok=True)\n    records = [r for r in records if r[\'criterion\'] in CRITERIA]\n    report=build_report(records,read_ratings(ratings_path))\n    (result_dir/\'heim_pilot_report.json\').write_text(json.dumps(report,ensure_ascii=False,indent=2,allow_nan=False)+\'\\n\',encoding=\'utf-8\')\n    with (result_dir/\'per_image.jsonl\').open(\'w\',encoding=\'utf-8\') as stream:\n        for row in records:\n            stream.write(json.dumps(row,ensure_ascii=False,allow_nan=False)+\'\\n\')\n    with (result_dir/\'summary.csv\').open(\'w\',newline=\'\',encoding=\'utf-8-sig\') as stream:\n        writer=csv.DictWriter(stream,fieldnames=[\'criterion\',\'name\',\'status\',\'images_planned\',\'images_readable\',\n            \'failed_missing_or_mismatch\',\'mean_clip_score\',\'mean_aesthetics_predictor\',\'human_ratings\',\'additional\'])\n        writer.writeheader()\n        for criterion,details in report[\'criteria\'].items():\n            additional={key:value for key,value in details.items() if key not in writer.fieldnames}\n            writer.writerow({\'criterion\':criterion, **{key:details.get(key) for key in writer.fieldnames if key in details},\n                \'human_ratings\':json.dumps(details[\'human_ratings\'],ensure_ascii=False),\n                \'additional\':json.dumps(additional,ensure_ascii=False)})\n    return report\n'
(DATA_ROOT/'heim_evaluate.py').write_text(EVALUATOR_SOURCE, encoding='utf-8')
import importlib
import heim_evaluate as he
he = importlib.reload(he)
print('Đã giải nén:',DATA_ROOT)

## 3. Kiểm tra dữ liệu
Mẫu lỗi, thiếu ảnh hoặc metadata không khớp được ghi riêng. Không thay các mẫu đó bằng điểm 0. Chọn model tương ứng với tên model lúc sinh ảnh.

In [ ]:
MODEL_ID = 'muse-image'  # Đổi khi đánh giá model khác
records, model_dir = he.load_records(DATA_ROOT,MODEL_ID)
from collections import Counter
for group,label in he.CRITERIA.items():
    items=[r for r in records if r['criterion']==group]
    print(label,':',dict(Counter(r['status'] for r in items)) if items else 'chưa có dữ liệu')
print('Tổng số ảnh theo kế hoạch:',len(records))

## 4. Chấm metric tự động
Tính CLIPScore và LAION aesthetic cho các nhóm đang chọn; kiểm tra tỷ lệ ảnh và tổng hợp thời gian API khi tạo báo cáo. Phong cách và các mục chấm người lấy từ CSV, không được tự thay bằng điểm CLIP. Lần đầu chạy cần tải trọng số CLIP ViT-L/14 và LAION.


In [ ]:
scorer = he.HeimClip()
print('Thiết bị:',scorer.device)
records = he.score_records(records,scorer)
print('Đã tính xong. Lỗi metric:',sum('metric_error' in r for r in records))

## 5. Tải lại phiếu đã chấm
Chạy ô dưới và chọn file `HELM/results/human_ratings.csv` trên máy, đã chấm bằng tool. Colab nhận file qua hộp thoại upload; không đọc trực tiếp đường dẫn local này.

Notebook kiểm tra điểm và gộp với metric tự động để tạo báo cáo JSON/CSV. Sau đó chạy ô **Cập nhật và tải kết quả**. Khi thay file điểm người, chỉ cần chạy lại hai ô này, không cần chạy lại CLIP.


In [ ]:
from google.colab import files

print('Chọn file HELM/results/human_ratings.csv đã chấm trên máy.')
uploaded_ratings = files.upload()
csvs = [name for name in uploaded_ratings if name.lower().endswith('.csv')]
if len(uploaded_ratings) != 1 or len(csvs) != 1:
    raise ValueError('Hãy chọn đúng một file CSV chấm người.')

RESULTS = Path('/content/heim_evaluation_results')
RESULTS.mkdir(exist_ok=True)
RATINGS = RESULTS/'human_ratings.csv'
candidate = RESULTS/'human_ratings_upload.csv'
candidate.write_bytes(uploaded_ratings[csvs[0]])
human_rows = he.read_ratings(candidate)
candidate.replace(RATINGS)

report = he.save_report(records, RATINGS, RESULTS)
import pandas as pd
summary = pd.read_csv(RESULTS/'summary.csv')
display(summary[['criterion','name','status','images_readable','mean_clip_score','mean_aesthetics_predictor','human_ratings']])
print('Đã nhập số dòng chấm người:', len(human_rows))
print('Kết quả tỷ lệ:', report['criteria']['13_aspect_ratio'].get('overall'))
print('Phiếu chấm:', RATINGS)


## 6. Cập nhật và tải kết quả
Chạy sau ô **Tải lại phiếu đã chấm** để tổng hợp điểm người cùng metric tự động và tải `heim_evaluation_results.zip` về máy.


In [ ]:
report=he.save_report(records,RATINGS,RESULTS)
display(pd.read_csv(RESULTS/'summary.csv')[['criterion','name','status','images_readable','human_ratings']])
import shutil
result_zip=shutil.make_archive('/content/heim_evaluation_results','zip',RESULTS)
from google.colab import files
files.download(result_zip)